# Context Breach TRL / GRPO Training Notebook

This notebook is the Colab-ready training path for **Context Breach: Contagion and Oversight in Multi-Agent Enterprise Systems**.

It does four things:

1. Installs the project and training dependencies.
2. Verifies the OpenEnv environment and baseline behavior.
3. Runs a TRL-compatible environment dry run.
4. Starts GRPO training with `environment_factory` so the model learns to call meaningful tools such as `inspect_artifact`, `quarantine_source`, `ask_verifier`, and `finalize_response`.

Use a GPU runtime in Colab before running the training cell.

## 1. Clone Or Open The Project

If this notebook is running inside the repo, skip the clone command. If running in a fresh Colab, replace the placeholder URL with your GitHub or Hugging Face Space repository URL.

In [ ]:
# Fresh Colab option:
# !git clone https://huggingface.co/spaces/YOUR_USERNAME/context-breach-env
# %cd context-breach-env

# If the notebook is already opened from the project root, this should show README.md and openenv.yaml.
!pwd
!ls -la | head

## 2. Install Dependencies

The environment itself is lightweight. TRL/Transformers may take a few minutes to install on Colab.

In [ ]:
!pip -q install -e .
!pip -q install -r requirements-training.txt

## 3. Validate Environment And Baselines

These commands prove the environment runs, OpenEnv packaging is valid, and the reward model separates unsafe from guarded behavior.

In [ ]:
!python3 -m pytest
!openenv validate --verbose
!python3 scripts/evaluate_baseline.py --policy naive --episodes 3
!python3 scripts/evaluate_baseline.py --policy guarded --episodes 3

## 4. Generate Baseline Result Plots

These plots are useful for the README and judging demo.

In [ ]:
!python3 scripts/generate_results.py

from IPython.display import Image, display
display(Image('results/reward_by_policy.png'))
display(Image('results/leakage_by_policy.png'))
display(Image('results/contamination_by_policy.png'))

## 5. TRL Environment Dry Run

This runs one safe tool trajectory using the same wrapper that GRPO uses. It should finish with `final_reward: 12.3` and `done: true`.

In [ ]:
!python3 scripts/train_trl_grpo.py --dry-run

## 6. Start GRPO Training

This is the minimum TRL training path for the hackathon. Start small first. After it runs, increase `--episodes`, `--max-steps`, and model size if compute allows.

Recommended first model: `Qwen/Qwen2.5-0.5B-Instruct` for a quick smoke run. If compute is available, try a stronger instruct model later.

In [ ]:
!python3 scripts/train_trl_grpo.py \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --episodes 60 \
  --max-steps 30 \
  --num-generations 4 \
  --gradient-accumulation-steps 4 \
  --max-completion-length 2048 \
  --output-dir outputs/context-breach-grpo

## 7. Demo Trace

Use this for the final pitch: it shows the unsafe baseline contaminating the team and the guarded policy containing the attack.

In [ ]:
!python3 scripts/generate_demo_trace.py

## Submission Notes

Before final submission, add these links to `README.md`:

- Hugging Face Space link
- This Colab notebook link
- Mini-video, mini-blog, or slide deck link

Minimum plots to include:

- Average reward
- Task success under attack
- Leakage rate
- Contamination depth or containment rate